In [1]:
import requests
import pandas as pd
import uuid

# Basis configuratie gebaseerd op de technische documentatie
BASE_URL = "https://api.ah.nl"
STORE_ID = "1558"  # Dit is het unieke ID voor AH Woenselse Markt Eindhoven

# Verplichte headers om de AH app na te bootsen
headers = {
    "User-Agent": "Appie/9.28 (iPhone17,3; iPhone; CPU OS 26_1 like Mac OS X)",
    "x-application": "AHWEBSHOP",
    "x-clientname": "appie-ios",
    "x-": "9.28",
    "x-fraud-detection-installation-id": str(uuid.uuid4()), # Een unieke ID per sessie
    "Content-Type": "application/json",
    "Accept": "application/json"
}


In [2]:
def get_anonymous_token():
    auth_url = f"{BASE_URL}/mobile-auth/v1/auth/token/anonymous"
    payload = {"clientId": "appie-ios"}
    
    response = requests.post(auth_url, json=payload, headers=headers)
    response.raise_for_status() # Geeft een foutmelding als het misgaat
    
    token_data = response.json()
    return token_data['access_token']

# Activeer de sleutel voor alle volgende verzoeken
access_token = get_anonymous_token()
headers["Authorization"] = f"Bearer {access_token}"
print("Handshake succesvol: Token opgehaald.")

Handshake succesvol: Token opgehaald.


In [7]:
bargain_query = """
query GetBargains($storeId: String!) {
  bargainItems(storeId: $storeId) {
    categoryTitle  # <--- Deze voegt de categorie (zoals Vlees) toe
    product {
      title
      brand
      salesUnitSize
    }
    bargainPrice {
      priceWas
      priceNow
    }
    markdown {
      markdownPercentage
      markdownExpirationDate
    }
    stock
  }
}
"""

def fetch_laatste_kans(store_id):
    url = f"{BASE_URL}/graphql"
    
    graphql_headers = headers.copy()
    graphql_headers.update({
        "x-apollo-operation-name": "GetBargains",
        "x-apollo-operation-type": "query",
        "apollographql-client-name": "nl.ah.Appie-apollo-ios",
        "apollographql-client-version": "9.28-260102201630"
    })
    
    payload = {
        'query': bargain_query, 
        'variables': {'storeId': store_id},
        'operationName': 'GetBargains'
    }
    
    response = requests.post(url, json=payload, headers=graphql_headers)
    data = response.json()
    
    if 'errors' in data:
        print(" GraphQL Foutmelding gevonden:")
        for error in data['errors']:
            print(f" - {error.get('message')}")
        return None
        
    return data.get('data', {}).get('bargainItems')

# Haal de ruwe data opnieuw op
raw_items = fetch_laatste_kans(STORE_ID)

if raw_items:
    print(f"Succes! {len(raw_items)} producten gevonden op de Woenselse Markt.")
else:
    print("Geen data ontvangen. Controleer de output hierboven.")

Succes! 173 producten gevonden op de Woenselse Markt.


In [9]:
# Gebruik json_normalize om geneste velden (zoals product.title) plat te slaan
df_koopjes = pd.json_normalize(raw_items)

# Optioneel: Kolomnamen opschonen voor gemak
df_koopjes.columns = [c.replace('product.', '').replace('bargainPrice.', '').replace('markdown.', '') for c in df_koopjes.columns]

# Sorteer op de hoogste korting
# df = df.sort_values(by='markdownPercentage', ascending=False)

# Toon de live status
display(df_koopjes.head(10))

,categoryTitle,stock,priceWas,priceNow,markdownPercentage,markdownExpirationDate,title,brand,salesUnitSize
0,"Groente, aardappelen",2,1.99,1.19,40,2026-01-26,Bieze Rauwkost komkommer,Bieze,250 g
1,"Groente, aardappelen",5,1.39,1.04,25,2026-01-26,AH Rucola slamelange,AH,75 g
2,"Groente, aardappelen",5,1.69,1.27,25,2026-01-26,AH Aardappelen voor de stamppot,AH,1 kg
3,"Groente, aardappelen",4,1.69,1.27,25,2026-01-26,AH Hutspot,AH,500 g
4,"Groente, aardappelen",4,2.99,2.24,25,2026-01-26,AH Roerbakgroente Italiaans fijngesneden,AH,400 g
5,"Groente, aardappelen",3,1.99,1.49,25,2026-01-26,AH Taugé grootverpakking,AH,250 g
6,"Groente, aardappelen",3,2.39,1.79,25,2026-01-26,AH Biologisch Vastkokende aardappelen,AH Biologisch,1 kg
7,"Groente, aardappelen",2,3.99,1.79,25,2026-01-26,AH Stoomgroente bloemkool marscarponesaus,AH,450 g
8,"Groente, aardappelen",2,1.59,1.19,25,2026-01-26,AH Eikenblad slamelange,AH,100 g
9,"Groente, aardappelen",2,4.49,3.37,25,2026-01-26,AH Pompoensoep verspakket,AH,6 pers | 35 min


In [150]:


def get_df_bonus():
    """Haalt landelijke bonus op en geeft een schoon pandas DataFrame terug"""
    token = get_ah_token()
    auth_headers = {**HEADERS, "Authorization": f"Bearer {token}"}
    
    # GraphQL query gebaseerd op het schema: bonusPromotions -> products -> priceV2
    query = """
    query GetNationalBonus {
      bonusPromotions {
        title
        products {
          title
          brand
          priceV2 {
            now { amount }
            was { amount }
            discount { description }
          }
        }
      }
    }
    """
    
    response = requests.post(f"{BASE_URL}/graphql", json={"query": query}, headers=auth_headers)
    response.raise_for_status()
    raw_data = response.json().get('data', {}).get('bonusPromotions', [])
    
    if not raw_data:
        return pd.DataFrame()

    # Transformatie: we 'flatten' de producten uit de promotiegroepen
    df = pd.json_normalize(
        raw_data, 
        record_path=['products'], 
        meta=['title'],
        record_prefix='product_',
        meta_prefix='promo_'
    )
    
    # Selectie en hernoemen van kolommen voor een schoon resultaat
    mapping = {
        'product_title': 'Product',
        'product_brand': 'Merk',
        'product_priceV2.now.amount': 'Prijs_Nu',
        'product_priceV2.was.amount': 'Prijs_Was',
        'product_priceV2.discount.description': 'Bonus_Tekst',
        'promo_title': 'Categorie'
    }
    
    # Alleen de kolommen die we willen hebben
    df_bonus = df.rename(columns=mapping)[list(mapping.values())]
    
    # Kleine extra opschoning: bereken het voordeel in euro's
    df_bonus['Korting_Euro'] = (df_bonus['Prijs_Was'] - df_bonus['Prijs_Nu']).round(2)
    
    return df_bonus

# --- UITVOERING ---
df_bonus = get_df_bonus()
# Check het resultaat
if not df_bonus.empty:
    print(f"✅ Gelukt! {len(df_bonus)} bonusitems geladen.")
    # Sorteer op de hoogste korting in euro's
    df_bonus = df_bonus.sort_values(by='Korting_Euro', ascending=False)
    print(df_bonus.head(10))
else:
    print("❌ Geen bonusdata gevonden.")

✅ Gelukt! 3515 bonusitems geladen.
                          Product   Merk  Prijs_Nu  Prijs_Was  Bonus_Tekst  \
3043         Nomad Fleece dames L  Nomad     19.99      39.98  50% korting   
3037    Nomad Fleece heren maat M  Nomad     19.99      39.98  50% korting   
3038   Nomad Fleece heren maat XL  Nomad     19.99      39.98  50% korting   
3039    Nomad Fleece heren maat L  Nomad     19.99      39.98  50% korting   
3040   Nomad Fleece dames maat XL  Nomad     19.99      39.98  50% korting   
3041    Nomad Fleece dames maat S  Nomad     19.99      39.98  50% korting   
3042    Nomad Fleece dames maat M  Nomad     19.99      39.98  50% korting   
3036  Nomad Fleece heren maat XXL  Nomad     19.99      39.98  50% korting   
3014     Nomad Ski handschoen S/M  Nomad     18.99      37.98  50% korting   
3016         Nomad Ski wanten S/M  Nomad     18.99      37.98  50% korting   

                Categorie  Korting_Euro  
3043  Nomad Thermokleding         19.99  
3037  Nomad Thermokled

In [151]:
df_bonus

,Product,Merk,Prijs_Nu,Prijs_Was,Bonus_Tekst,Categorie,Korting_Euro
3043,Nomad Fleece dames L,Nomad,19.99,39.98,50% korting,Nomad Thermokleding,19.99
3037,Nomad Fleece heren maat M,Nomad,19.99,39.98,50% korting,Nomad Thermokleding,19.99
3038,Nomad Fleece heren maat XL,Nomad,19.99,39.98,50% korting,Nomad Thermokleding,19.99
3039,Nomad Fleece heren maat L,Nomad,19.99,39.98,50% korting,Nomad Thermokleding,19.99
3040,Nomad Fleece dames maat XL,Nomad,19.99,39.98,50% korting,Nomad Thermokleding,19.99
...,...,...,...,...,...,...,...
3510,Dr. van der Hoog Moddermasker dode zee,Dr. van der Hoog,2.99,NaN,1 + 1 gratis,Biodermal en Dr. van der Hoog gezichtsverzorging,NaN
3511,Biodermal Dag- & nachtcrème,Biodermal,29.99,NaN,1 + 1 gratis,Biodermal en Dr. van der Hoog gezichtsverzorging,NaN
3512,Biodermal Dagcreme p-cl-e crème,Biodermal,41.99,NaN,1 + 1 gratis,Biodermal en Dr. van der Hoog gezichtsverzorging,NaN
3513,Biodermal Reinigingsmousse,Biodermal,19.99,NaN,1 + 1 gratis,Biodermal en Dr. van der Hoog gezichtsverzorging,NaN


In [153]:
def get_df_bonus():
    token = get_ah_token()
    auth_headers = {**HEADERS, "Authorization": f"Bearer {token}"}
    
    # GraphQL query: 'mainCategory' vervangen door 'category'
    query = """
    query GetNationalBonus {
      bonusPromotions {
        title
        category  # De categorie van de hele promotiegroep
        products {
          title
          brand
          category # De categorie van het specifieke product
          priceV2 {
            now { amount }
            was { amount }
            discount { description }
          }
        }
      }
    }
    """
    
    response = requests.post(f"{BASE_URL}/graphql", json={"query": query}, headers=auth_headers)
    response.raise_for_status()
    raw_data = response.json().get('data', {}).get('bonusPromotions', [])
    
    if not raw_data:
        return pd.DataFrame()

    # We flatten de producten en nemen de categorieën mee
    df = pd.json_normalize(
        raw_data, 
        record_path=['products'], 
        meta=['title', 'category'], # 'category' van de promo toevoegen als meta
        record_prefix='product_',
        meta_prefix='promo_'
    )
    
    # Mapping: Gebruik 'product_category' voor de meest specifieke indeling
    mapping = {
        'product_title': 'Product',
        'product_brand': 'Merk',
        'product_category': 'Categorie',  # Dit veld komt nu wel door
        'product_priceV2.now.amount': 'Prijs_Nu',
        'product_priceV2.was.amount': 'Prijs_Was',
        'product_priceV2.discount.description': 'Bonus_Tekst',
        'promo_title': 'Bonus_Groep'
    }
    
    df_bonus = df.rename(columns=mapping)[list(mapping.values())]
    
    # Berekeningen
    df_bonus['Prijs_Nu'] = pd.to_numeric(df_bonus['Prijs_Nu'], errors='coerce')
    df_bonus['Prijs_Was'] = pd.to_numeric(df_bonus['Prijs_Was'], errors='coerce')
    df_bonus['Korting_Pct'] = ((df_bonus['Prijs_Was'] - df_bonus['Prijs_Nu']) / df_bonus['Prijs_Was'] * 100).round(0)
    
    return df_bonus

In [155]:
df_bonus = get_df_bonus()

In [158]:
# 1. Normaliseer de titels zoals we al deden
df_bonus['Product_clean'] = df_bonus['Product'].str.lower().str.strip()
df_koopjes['title_clean'] = df_koopjes['title'].str.lower().str.strip()

# 2. De Merge
df_double_deals = pd.merge(
    df_bonus, 
    df_koopjes, 
    left_on='Product_clean', 
    right_on='title_clean', 
    how='inner'
)

# 3. FIX: Zet prijzen om naar getallen (errors='coerce' maakt van tekst NaN)
df_double_deals['priceNow'] = pd.to_numeric(df_double_deals['priceNow'], errors='coerce')
df_double_deals['Prijs_Was'] = pd.to_numeric(df_double_deals['Prijs_Was'], errors='coerce')

# 4. Bereken nu de korting (met beveiliging tegen delen door nul)
df_double_deals['Totale_Korting_Percentage'] = (1 - (df_double_deals['priceNow'] / df_double_deals['Prijs_Was'])) * 100

# 5. Filter en toon resultaat
sniper_resultaat = df_double_deals[[
    'Product', 
    'Bonus_Tekst', 
    'markdownPercentage', 
    'Prijs_Was', 
    'priceNow', 
    'Totale_Korting_Percentage'
]].dropna(subset=['Totale_Korting_Percentage']) # Haal ongeldige berekeningen eruit


df_double_deals['Totale_Korting_Percentage'] = df_double_deals['Totale_Korting_Percentage'].round(2)
display(sniper_resultaat.sort_values(by='Totale_Korting_Percentage', ascending=False))

,Product,Bonus_Tekst,markdownPercentage,Prijs_Was,priceNow,Totale_Korting_Percentage
17,Johma Oudekaas-pesto salade,30% korting,25,3.69,1.94,47.425474
0,Friesche Vlag Barista haver,NaN,40,2.69,1.61,40.148699
14,AH Scharrel kipfilet blokjes,1 euro korting,25,5.49,3.37,38.615665
1,Optimel Drinkyoghurt mango-passievrucht,NaN,25,1.59,1.19,25.157233
2,L'Atelier Melkchocoladereep hazelnoot & rozijn,NaN,25,4.39,3.29,25.056948


In [159]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv(override=True)

# Haal de key op en maak hem direct schoon
raw_key = os.getenv("GEMINI_API_KEY")
if raw_key:
    # Verwijder spaties, quotes en witregels
    api_key = raw_key.strip().strip('"').strip("'")
    
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel("gemma-3-27b-it")    
    try:
        # Test de verbinding
        response = model.generate_content("Hoi Chef!")
        print("✅ Verbinding geslaagd! De motor draait.")
    except Exception as e:
        print(f"❌ Google zegt nog steeds nee: {e}")
else:
    print("❌ Sleutel niet gevonden in .env")

✅ Verbinding geslaagd! De motor draait.


In [141]:
df_koopjes['categoryTitle'].unique()

array(['Groente, aardappelen', 'Fruit, verse sappen',
       'Maaltijden, salades', 'Vlees', 'Vis',
       'Vegetarisch, vegan en plantaardig', 'Vleeswaren', 'Kaas',
       'Zuivel, eieren', 'Bakkerij', 'Borrel, chips, snacks',
       'Pasta, rijst, wereldkeuken', 'Soepen, sauzen, kruiden, olie',
       'Koek, snoep, chocolade', 'Ontbijtgranen, beleg', 'Tussendoortjes',
       'Koffie, thee', 'Frisdrank, sappen, water'], dtype=object)

In [ ]:
df_